In [5]:
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    Attention,
    GlobalAveragePooling1D
)
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
X = np.load("../processed/X.npy")
y = np.load("../processed/y.npy")
subjects = np.load("../processed/subjects.npy")

print(X.shape)
print(y.shape)
print(subjects.shape)

(4635, 7, 297)
(4635,)
(4635,)


In [7]:
unique_subjects = np.unique(subjects)

subject_kfold = KFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

train_sub_idx, test_sub_idx = next(
    subject_kfold.split(unique_subjects)
)

train_subjects = unique_subjects[train_sub_idx]
test_subjects = unique_subjects[test_sub_idx]

print("Train Subjects:", len(train_subjects))
print("Test Subjects:", len(test_subjects))

print(
    "Intersection:",
    np.intersect1d(
        train_subjects,
        test_subjects
    )
)

Train Subjects: 92
Test Subjects: 11
Intersection: []


In [8]:
train_mask = np.isin(
    subjects,
    train_subjects
)

test_mask = np.isin(
    subjects,
    test_subjects
)

X_train = X[train_mask]
y_train = y[train_mask]

X_test = X[test_mask]
y_test = y[test_mask]

print(X_train.shape)
print(X_test.shape)

(4140, 7, 297)
(495, 7, 297)


In [9]:
scaler = StandardScaler()

X_train_flat = X_train.reshape(
    -1,
    297
)

X_test_flat = X_test.reshape(
    -1,
    297
)

X_train_flat = scaler.fit_transform(
    X_train_flat
)

X_test_flat = scaler.transform(
    X_test_flat
)

X_train = X_train_flat.reshape(
    X_train.shape
)

X_test = X_test_flat.reshape(
    X_test.shape
)

print(X_train.shape)
print(X_test.shape)

(4140, 7, 297)
(495, 7, 297)


In [10]:
inputs = Input(
    shape=(7,297)
)

x = LSTM(
    256,
    return_sequences=True
)(inputs)

x = Dropout(0.2)(x)

x = LSTM(
    256,
    return_sequences=True
)(x)

x = Dropout(0.1)(x)

x = LSTM(
    256,
    return_sequences=True
)(x)

x = Dropout(0.2)(x)

attention_output = Attention()(
    [x, x]
)

x = GlobalAveragePooling1D()(
    attention_output
)

outputs = Dense(
    1,
    activation="sigmoid"
)(x)

model = Model(
    inputs,
    outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 7, 297)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 7, 256)    │    567,296 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 7, 256)    │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 7, 256)    │    525,312 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 7, 256)    │          0 │ lstm_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ (None, 7, 256)    │    525,312 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 7, 256)    │          0 │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, 7, 256)    │          0 │ dropout_2[0][0],  │
│ (Attention)         │                   │            │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ attention[0][0]   │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 1)         │        257 │ global_average_p… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,618,177 (6.17 MB)

 Trainable params: 1,618,177 (6.17 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [12]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [13]:
history = model.fit(
    X_train,
    y_train,
    validation_split=0.1,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop]
)

Epoch 1/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 14s 72ms/step - accuracy: 0.5899 - loss: 0.6614 - val_accuracy: 0.5580 - val_loss: 0.6851
Epoch 2/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 69ms/step - accuracy: 0.6817 - loss: 0.5755 - val_accuracy: 0.5870 - val_loss: 0.7325
Epoch 3/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 69ms/step - accuracy: 0.7627 - loss: 0.4674 - val_accuracy: 0.6135 - val_loss: 0.7083
Epoch 4/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 67ms/step - accuracy: 0.8242 - loss: 0.3570 - val_accuracy: 0.6232 - val_loss: 0.8617
Epoch 5/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 8s 68ms/step - accuracy: 0.8910 - loss: 0.2541 - val_accuracy: 0.6353 - val_loss: 1.0488
Epoch 6/100
117/117 ━━━━━━━━━━━━━━━━━━━━ 10s 66ms/step - accuracy: 0.9283 - loss: 0.1682 - val_accuracy: 0.6643 - val_loss: 1.2778


In [14]:
pred = model.predict(
    X_test
)

pred = (
    pred > 0.5
).astype(int)

print(
    classification_report(
        y_test,
        pred
    )
)

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 56ms/step
              precision    recall  f1-score   support

           0       0.65      0.48      0.55       254
           1       0.57      0.72      0.64       241

    accuracy                           0.60       495
   macro avg       0.61      0.60      0.59       495
weighted avg       0.61      0.60      0.59       495

